In [2]:
import time
from pathlib import Path

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_DIR = Path("/home/eleroy/proj-dirs/fast-o3-iv/data/external/NCAR/CESM2/ssp370")
BASE_URL = "https://osdf-director.osg-htc.org/ncar/gdex/d651056/CESM2-LE/atm/proc/tseries/month_1"

VARIABLE = "V"

MEMBERS = [
    "LE2-1001.001",
    "LE2-1021.002",
    "LE2-1041.003",
    "LE2-1061.004",
    "LE2-1081.005",
    "LE2-1101.006",
    "LE2-1121.007",
    "LE2-1141.008",
    "LE2-1161.009",
    "LE2-1181.010",
    "LE2-1231.001",
    "LE2-1231.002",
    "LE2-1231.003",
    "LE2-1231.004",
    "LE2-1231.005",
    "LE2-1231.006",
    "LE2-1231.007",
]

DATE_RANGES = [
    "201501-202412",
    "202501-203412",
    "203501-204412",
    "204501-205412",
]

session = requests.Session()
retry = Retry(
    total=8,
    connect=8,
    read=8,
    backoff_factor=2.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)
session.headers.update({
    "User-Agent": "fast-o3-iv-downloader/1.0",
    "Accept-Encoding": "identity",
})

def build_filename(member: str, var: str, date_range: str) -> str:
    member_a, member_b = member.split(".")  # LE2-1001 / 001
    return (
        f"b.e21.BSSP370cmip6.f09_g17."
        f"{member_a}.{member_b}.cam.h0.{var}.{date_range}.nc"
    )

def build_url(member: str, var: str, date_range: str) -> str:
    fname = build_filename(member, var, date_range)
    return f"{BASE_URL}/{var}/{fname}"

def download_with_resume(url: str, out_path: Path, attempts: int = 8) -> None:
    tmp_path = out_path.with_suffix(out_path.suffix + ".part")

    for i in range(1, attempts + 1):
        try:
            start = tmp_path.stat().st_size if tmp_path.exists() else 0
            headers = {"Range": f"bytes={start}-"} if start > 0 else {}

            with session.get(
                url,
                stream=True,
                timeout=(30, 600),
                headers=headers,
                allow_redirects=True,
            ) as r:
                if start > 0 and r.status_code == 200:
                    tmp_path.unlink(missing_ok=True)
                    start = 0

                r.raise_for_status()

                mode = "ab" if start > 0 else "wb"
                with open(tmp_path, mode) as f:
                    for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                        if chunk:
                            f.write(chunk)

            tmp_path.replace(out_path)
            return

        except Exception as e:
            print(f"Attempt {i}/{attempts} failed for {url}: {e}")
            if i == attempts:
                raise
            time.sleep(min(60, 2**i))

for member in MEMBERS:
    out_dir = BASE_DIR / member / VARIABLE
    out_dir.mkdir(parents=True, exist_ok=True)

    for date_range in DATE_RANGES:
        fname = build_filename(member, VARIABLE, date_range)
        url = build_url(member, VARIABLE, date_range)
        out_path = out_dir / fname

        if out_path.exists() and out_path.stat().st_size > 0:
            print(f"Skipping existing: {out_path}")
            continue

        print(f"Downloading {url} -> {out_path}")
        download_with_resume(url, out_path)
        print("done")

Attempt 1/8 failed for https://osdf-director.osg-htc.org/ncar/gdex/d651056/CESM2-LE/atm/proc/tseries/month_1/V/b.e21.BSSP370cmip6.f09_g17.LE2-1001.001.cam.h0.V.201501-202412.nc: ('Connection broken: IncompleteRead(392167424 bytes read, 272599175 more expected)', IncompleteRead(392167424 bytes read, 272599175 more expected))
Attempt 2/8 failed for https://osdf-director.osg-htc.org/ncar/gdex/d651056/CESM2-LE/atm/proc/tseries/month_1/V/b.e21.BSSP370cmip6.f09_g17.LE2-1001.001.cam.h0.V.201501-202412.nc: ('Connection broken: IncompleteRead(268435456 bytes read, 10455175 more expected)', IncompleteRead(268435456 bytes read, 10455175 more expected))
done
Attempt 1/8 failed for https://osdf-director.osg-htc.org/ncar/gdex/d651056/CESM2-LE/atm/proc/tseries/month_1/V/b.e21.BSSP370cmip6.f09_g17.LE2-1001.001.cam.h0.V.202501-203412.nc: ('Connection broken: IncompleteRead(236978176 bytes read, 427623464 more expected)', IncompleteRead(236978176 bytes read, 427623464 more expected))
Attempt 2/8 failed 

KeyboardInterrupt: 